### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.

In [ ]:
import os, sys
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group = f"lab-{deployment_name}"

# The hosted demo UI (host_demo_ui = True) registers an application in Microsoft Entra ID, which lives outside the resource group
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group} --query properties.outputs.demoUiAppId.value -o tsv",
                   print_command_to_run=False)
demo_ui_app_id = output.text.strip() if output.success else ""
if demo_ui_app_id:
    utils.run(f"az ad app delete --id {demo_ui_app_id}", f"Deleted the demo UI app registration {demo_ui_app_id}",
              f"Failed to delete the demo UI app registration {demo_ui_app_id}")

# The "over budget, switched off" loop can create a custom role definition. Role definitions live at the subscription level,
# so deleting the resource group (which removes the Logic App's role assignment) doesn't remove it: read it before, delete it after.
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group} --query properties.outputs.budgetSuspend.value -o json",
                   print_command_to_run=False)
budget_suspend = output.json_data if output.success and isinstance(output.json_data, dict) else {}

utils.cleanup_resources(deployment_name, resource_group_name=resource_group)

if budget_suspend.get("enabled") and budget_suspend.get("customRole"):
    role_id = budget_suspend["roleDefinitionId"]
    # DELETE by the full id: 'az role definition delete --name <guid>' can exit 0 without deleting when its lookup comes back empty
    utils.run(f"az rest --method delete --url https://management.azure.com{role_id}?api-version=2022-04-01",
              f"Deleted the custom role definition {role_id.split('/')[-1]}", f"Failed to delete the custom role definition {role_id}")